# Jev: typed AI decisions with TypeSafe

A hands-on introduction to TypeSafe's **Jev System One model** using the official Python SDK.

You will set up an account and API key, create a `uv` environment, load credentials from `.env`, discover models, and try **Noul**, **Choice**, and **Score**. Then you will combine those decisions in a small support-ticket workflow.

**Checked against official documentation on October 1, 2026.** Code cells are ready for live execution; this notebook contains no recorded API results. Each evaluation cell sends data to TypeSafe and may incur usage charges. All examples use fictional data.

Run the notebook from top to bottom after completing sections 1–2.


## 1. Sign up and obtain an API key

1. Open [TypeSafe Console](https://console.typesafe.ai/). The current login page offers **Continue with Google** or an email address followed by **Continue**. Use either path and finish the account/onboarding prompts.
2. In the authenticated dashboard, locate the API-key controls and create or obtain a key. TypeSafe's [quick start](https://docs.typesafe.ai/introduction/quickstart) directs users to the dashboard for keys; exact menu labels may change.
3. Check your account's available API access and usage/billing settings before running the examples. If key creation is unavailable, follow the console's access instructions or contact TypeSafe.
4. Save the key locally in `.env` as shown below. Do not paste it into notebook cells or output.

Optional: use the console Playground to explore questions before writing code. You do not need an OpenAI API key for this notebook.


## 2. Set up the environment with uv

The accompanying project bundle includes `pyproject.toml`, `uv.lock`, `.env.example`, and this notebook. Extract it and open a terminal in the `jev_showcase` folder.

Install [uv](https://docs.astral.sh/uv/getting-started/installation/) if necessary:

```bash
# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh
```

```powershell
# Windows PowerShell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

Restart your terminal, then:

```bash
uv --version
uv sync
uv run jupyter lab
```

`uv sync` uses the bundled `uv.lock` and installs dependencies into `.venv`. Without a lockfile, it resolves dependencies and writes one. Keep the generated lockfile to reproduce the resolved versions; use `uv sync --locked` on subsequent machines.

In JupyterLab open `jev_showcase.ipynb` and choose the project's Python kernel. In VS Code, install the Python and Jupyter extensions and select `.venv/bin/python` (Windows: `.venv\Scripts\python.exe`). If you need a named kernel:

```bash
uv run python -m ipykernel install --user --name jev-showcase --display-name "Jev Showcase"
```

### If you downloaded only the notebook

Put it in a new folder and create `pyproject.toml` alongside it with:

```toml
[project]
name = "jev-showcase"
version = "0.1.0"
requires-python = ">=3.11,<3.14"
dependencies = [
  "typesafe-sdk",
  "python-dotenv>=1.0,<2",
  "jupyterlab>=4,<5",
  "ipykernel>=6,<8",
  "matplotlib>=3.8,<4",
]

[tool.uv]
package = false
```

Then run `uv sync` and `uv run jupyter lab` from that folder.

### Create .env

Copy `.env.example` to `.env` (macOS/Linux: `cp .env.example .env`; PowerShell: `Copy-Item .env.example .env`). If using the standalone notebook, create `.env` yourself. Edit it locally:

```dotenv
TYPESAFE_API_KEY=replace_with_your_real_key
TYPESAFE_DEFAULT_MODEL=jev-latest
```

Add `.env`, `.venv/`, and `.ipynb_checkpoints/` to `.gitignore`. Keep `.env.example` as a key-free template. Restart the kernel after changing credentials.


## 3. Load credentials and create a client

The loader searches the working directory and its parents for `.env`. Existing shell variables take precedence (`override=False`). The key is checked without displaying it. A GPU is unnecessary because inference runs through the hosted API.


In [1]:
import os
import sys
from pathlib import Path
from time import perf_counter

from dotenv import load_dotenv
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
from typesafe_sdk import (
    TypeSafeClient, Noul, Choice, Score,
    TypeSafeAPIError, TypeSafeAPIConnectionError,
)

env_path = next(
    (folder / ".env" for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError("Create .env alongside the notebook, then rerun this cell.")
load_dotenv(env_path, override=False)
api_key = os.getenv("TYPESAFE_API_KEY", "").strip()
if not api_key or api_key == "replace_with_your_real_key":
    raise ValueError("Set TYPESAFE_API_KEY in .env, then restart the kernel.")
MODEL = os.getenv("TYPESAFE_DEFAULT_MODEL", "jev-latest").strip() or "jev-latest"
client = TypeSafeClient(api_key=api_key, model=MODEL, timeout=30.0)
print("Python:", sys.executable)
print("Credentials loaded; requested model:", MODEL)


Python: /Users/mlstudio/git_kmehrad/LLMs-private/jev_showcase/.venv/bin/python
Credentials loaded; requested model: jev-latest


## 4. Discover models available to your account

The SDK calls `GET /v1/models`. Use a returned name as the model identifier. `jev-latest` is a convenient alias; record the resolved model in evaluation logs because the alias can change over time.


In [2]:
models = client.models.list()
for model in models.models:
    print(f"{model.name}: {model.description} (released {model.release_date})")


jev-latest: The latest iteration of TypeSafe's System One Model: Jev (released 2026-09-10T18:38:01.391457+00:00)
jev-preview: A preview version of `jev-latest`: should be better in most ways (released 2026-09-10T18:39:06.057655+00:00)


## 5. Noul: a yes/no probability

A **Noul** returns `noul`, the probability of yes/true on a 0–1 scale. It does not include a separate `confidence` field. Name the answer for convenient lookup, and put the actual question in `instructions`.


In [3]:
message = "Our checkout has been failing since this morning. Customers cannot pay. Please help today."
noul_result = client.system_one(
    state=message,
    questions={
        "time_sensitive": Noul(
            instructions="Does the message describe a time-sensitive problem?",
            criteria={
                "true": "An active disruption or explicit deadline requires prompt attention.",
                "false": "A general inquiry with no immediate disruption or deadline.",
            },
        ),
    },
)
p_yes = noul_result.answers["time_sensitive"].noul
print(f"P(time-sensitive): {p_yes:.3f}")
print("Resolved model:", noul_result.model)


P(time-sensitive): 0.980
Resolved model: jev-1.13.0


## 6. Choice: classify into defined categories

A **Choice** returns a selected option, probabilities for all options, and `confidence`. Include `other` to cover messages outside your categories. The category definitions below are our example rubric.


In [4]:
department_question = Choice(
    instructions="Which department best matches the primary issue in the customer's message?",
    criteria={
        "billing": "Charges, invoices, refunds, or subscription payments.",
        "technical": "Broken functionality, outages, or integration errors.",
        "sales": "Product pricing, plan comparisons, or buying a new service.",
        "other": "Insufficient context or an issue outside the listed departments.",
    },
)
choice_result = client.system_one(
    state={"customer_message": "I see two charges for the same subscription. Can you investigate?"},
    questions={"department": department_question},
)
department = choice_result.answers["department"]
print("Selected:", department.choice)
print(f"Confidence: {department.confidence:.3f}")
display(department.probabilities)


Selected: billing
Confidence: 1.000


{'billing': 1.0, 'technical': 0.0, 'sales': 0.0, 'other': 0.0}

## 7. Score: evaluate an ordered rubric

A **Score** uses zero-based rubric levels. Its result is an expected value, so it can fall between levels. Here the levels mean 0 = routine, 1 = soon, 2 = immediate. This scale is defined by us; it is not a universal Jev scale.


In [5]:
urgency_question = Score(
    instructions="How urgently does the customer's described problem need attention?",
    criteria=[
        "Routine: informational request with no deadline or disruption.",
        "Soon: inconvenience or deadline within the next several days, without a complete outage.",
        "Immediate: active outage, inability to complete essential work, or explicit same-day deadline.",
    ],
)
score_result = client.system_one(
    state=message,
    questions={"urgency": urgency_question},
)
urgency = score_result.answers["urgency"]
print(f"Expected urgency: {urgency.score:.3f} on a 0–2 rubric")
print(f"Confidence: {urgency.confidence:.3f}")
display(urgency.legend)
display(urgency.probabilities)


Expected urgency: 2.000 on a 0–2 rubric
Confidence: 1.000


{0: 'Routine: informational request with no deadline or disruption.',
 1: 'Soon: inconvenience or deadline within the next several days, without a complete outage.',
 2: 'Immediate: active outage, inability to complete essential work, or explicit same-day deadline.'}

{0: 0.0, 1: 0.0, 2: 1.0}

## 8. Ask all three in one request

Each named question reads the same state independently. A question cannot read another question's answer; compose dependencies in Python after receiving the results. Structured state can carry the message and relevant records.


In [6]:
questions = {
    "department": department_question,
    "urgency": urgency_question,
    "refund_requested": Noul(
        instructions="Does the customer explicitly ask for a refund in customer_message?",
        criteria={
            "true": "The customer directly asks to have money returned.",
            "false": "No explicit request for money to be returned.",
        },
    ),
}
ticket = {
    "customer_message": "I was billed twice for this month. Please refund the duplicate charge today.",
    "account": {"plan": "business", "subscription_status": "active"},
}
started = perf_counter()
result = client.system_one(state=ticket, questions=questions)
elapsed = perf_counter() - started
print(result.model_dump_json(indent=2))
print(f"Observed wall-clock time: {elapsed:.3f}s (includes network and any retries)")
print("Resolved model:", result.model)
print("Token usage:", result.usage.model_dump())


{
  "model": "jev-1.13.0",
  "usage": {
    "input_tokens": 559,
    "output_tokens": 77
  },
  "answers": {
    "department": {
      "type": "choice",
      "choice": "billing",
      "confidence": 1.0,
      "probabilities": {
        "billing": 1.0,
        "technical": 0.0,
        "sales": 0.0,
        "other": 0.0
      }
    },
    "urgency": {
      "type": "score",
      "score": 2.0,
      "confidence": 1.0,
      "legend": {
        "0": "Routine: informational request with no deadline or disruption.",
        "1": "Soon: inconvenience or deadline within the next several days, without a complete outage.",
        "2": "Immediate: active outage, inability to complete essential work, or explicit same-day deadline."
      },
      "probabilities": {
        "0": 0.0,
        "1": 0.0,
        "2": 1.0
      }
    },
    "refund_requested": {
      "type": "noul",
      "noul": 0.99
    }
  }
}
Observed wall-clock time: 0.192s (includes network and any retries)
Resolved model: 

## 9. Inspect distributions

The plots show the actual response from the preceding cell. Choice/Score confidence summarizes distribution shape; it is not interchangeable with the probability of the selected option or a guarantee of correctness. Noul is a yes-probability.

For example, an expected score near 1 could come from certainty at level 1 or a mixture of levels 0 and 2. Inspect the full distribution when that distinction matters.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, name in zip(axes, ["department", "urgency"]):
    answer = result.answers[name]
    labels = list(answer.probabilities)
    ax.bar(labels, [answer.probabilities[label] for label in labels], color="#3578a8")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Probability")
    ax.set_title(f"{name} — confidence {answer.confidence:.2f}")
fig.tight_layout()
plt.show()


## 10. Combine decisions in code

Our illustrative policy sends uncertain classifications to review and flags likely refund requests. It only returns routing suggestions; it does not perform a refund. The numerical thresholds are teaching choices. Tune them on labeled examples using precision/recall, coverage, error costs, and calibration checks before deployment.


In [ ]:
def route_ticket(response, min_confidence=0.80, refund_low=0.20, refund_high=0.80):
    dept = response.answers["department"]
    urgent = response.answers["urgency"]
    refund_probability = response.answers["refund_requested"].noul
    if dept.confidence < min_confidence or dept.choice == "other":
        return {"route": "human_review", "reason": "Department needs review"}
    if urgent.confidence < min_confidence:
        return {"route": "human_review", "reason": "Urgency needs review"}
    if refund_low < refund_probability < refund_high:
        return {"route": "human_review", "reason": "Refund intent is uncertain"}
    return {
        "route": dept.choice,
        "priority": "high" if urgent.score >= 1.5 else "normal",
        "refund_review_required": refund_probability >= refund_high,
    }

display(route_ticket(result))


## 11. Apply the same rubric to several tickets (optional)

This cell makes **three additional evaluation requests**, sequentially. It reuses the same questions and reports model outputs and routing suggestions. A list passed as `state` would be one shared state, not an automatic batch of independent evaluations.


In [ ]:
sample_tickets = [
    {"id": "T001", "customer_message": "How much does the team plan cost?"},
    {"id": "T002", "customer_message": "All users are locked out and we have a demo today."},
    {"id": "T003", "customer_message": "Something seems wrong with my account; I am not sure what."},
]
rows = []
for sample in sample_tickets:
    response = client.system_one(state=sample, questions=questions)
    dept = response.answers["department"]
    rows.append({
        "id": sample["id"], "department": dept.choice,
        "department_confidence": round(dept.confidence, 3),
        "urgency": round(response.answers["urgency"].score, 3),
        "resolved_model": response.model,
        "suggestion": route_ticket(response),
    })
display(rows)


## 12. Error handling and troubleshooting

The SDK supplies retries for transient failures. Catch errors at the application boundary. The function below shows a compact pattern and is defined without making another API call. It omits raw error bodies because they may contain submitted data.


In [ ]:
def evaluate_with_diagnostics(state, questions):
    try:
        return client.system_one(state=state, questions=questions)
    except TypeSafeAPIConnectionError as exc:
        raise RuntimeError("Connection failed or timed out; check network and API status.") from exc
    except TypeSafeAPIError as exc:
        hints = {
            401: "Check TYPESAFE_API_KEY and restart the kernel after editing .env.",
            403: "Check account permissions and model access.",
            422: "Check the state, model, and question schema against the API reference.",
            429: "Rate limited; reduce request frequency after SDK retries are exhausted.",
            529: "Service overloaded; try again later.",
        }
        hint = hints.get(exc.status, "Check the API status and current documentation.")
        raise RuntimeError(f"TypeSafe HTTP {exc.status}: {hint}") from exc


| Symptom | Next step |
|---|---|
| `ModuleNotFoundError` | Run `uv sync` in the project folder and select its `.venv` kernel. |
| `.env` missing | Put `.env` alongside the notebook and start Jupyter from that folder. |
| Old key still used | Restart the kernel; existing environment variables take precedence over `.env`. |
| Model rejected | Rerun model discovery and use a returned identifier. |
| 422 validation error | Compare your request to the current API schema. |
| 429 or 529 | Reduce load or retry later; the SDK already retries transient failures. |

## 13. Clean up

Run this when finished. To make more requests afterward, recreate the client by rerunning section 3. Clear outputs before sharing a notebook that has processed private data.


In [ ]:
client.close()
print("Client closed.")


## References and next experiments

Official sources:

- [TypeSafe introduction](https://docs.typesafe.ai/introduction)
- [Console / signup](https://console.typesafe.ai/) and [quick start](https://docs.typesafe.ai/introduction/quickstart)
- [HTTP API reference](https://docs.typesafe.ai/api) and [OpenAPI schema](https://api.typesafe.ai/openapi.json)
- [Python client](https://docs.typesafe.ai/sdk/python/api/clients/sync) and [responses](https://docs.typesafe.ai/sdk/python/api/types/responses)
- [Confidence](https://docs.typesafe.ai/confidence) and [exceptions](https://docs.typesafe.ai/sdk/python/api/exceptions)
- [uv + Jupyter](https://docs.astral.sh/uv/guides/integration/jupyter/)

Try adding labeled tickets, comparing rubrics, and sweeping routing thresholds. For an LLM workflow, use separate narrow questions to judge relevance and factual support, then combine them in code. Verify supported input formats before extending this example to other modalities.

**Validation:** notebook syntax and workflow can be checked without credentials. Live predictions require your own API key; the distributed notebook has no fabricated model outputs.
